# Outil de gestion de bibliothèque

On commence ici un outil de gestion des emprunts d'une bibliothèque, avec l'ORM de SQLAlchemy sur une base SQLite : modéliser les livres, les membres et leurs emprunts, importer des données depuis un fichier YAML, interroger la base, puis exporter les emprunts en CSV.

Les étapes s'enchaînent : chaque solution définit ce que la suite utilise. Si vous êtes bloqué, exécutez la solution avant de passer à la suite.

## Mise en place

La cellule suivante écrit les données de la bibliothèque dans `bibliotheque.yaml` : des livres, des membres, et des emprunts datés, dont certains sont rendus (ils ont une date de `retour`). La suivante crée le moteur, sur le fichier `bibliotheque.db`, et la classe `Base` des modèles.

In [ ]:
%%writefile bibliotheque.yaml
livres:
  - titre: Vingt Mille Lieues sous les mers
    auteur: Jules Verne
  - titre: Voyage au centre de la Terre
    auteur: Jules Verne
  - titre: De la Terre à la Lune
    auteur: Jules Verne
  - titre: Le Tour du monde en quatre-vingts jours
    auteur: Jules Verne
  - titre: Michel Strogoff
    auteur: Jules Verne
  - titre: The Hound of the Baskervilles
    auteur: Arthur Conan Doyle
  - titre: Arsène Lupin, gentleman-cambrioleur
    auteur: Maurice Leblanc
membres:
  - John Doe
  - Jane Doe
  - Sherlock Holmes
  - Arsène Lupin
emprunts:
  - livre: Voyage au centre de la Terre
    membre: Jane Doe
    date: "2026-01-03"
    retour: "2026-01-24"
  - livre: Le Tour du monde en quatre-vingts jours
    membre: Sherlock Holmes
    date: "2026-01-05"
    retour: "2026-01-19"
  - livre: Michel Strogoff
    membre: Sherlock Holmes
    date: "2026-01-07"
  - livre: Arsène Lupin, gentleman-cambrioleur
    membre: Jane Doe
    date: "2026-01-20"
    retour: "2026-02-01"
  - livre: The Hound of the Baskervilles
    membre: Sherlock Holmes
    date: "2026-02-02"
  - livre: Voyage au centre de la Terre
    membre: John Doe
    date: "2026-02-10"

In [ ]:
from sqlalchemy import create_engine, select
from sqlalchemy.orm import DeclarativeBase, Session

engine = create_engine("sqlite:///bibliotheque.db")


class Base(DeclarativeBase):
  pass

## 1. Modéliser livres, membres et emprunts

Un membre emprunte plusieurs livres, et un livre est emprunté par plusieurs membres au fil du temps : c'est une relation plusieurs-à-plusieurs. Chaque emprunt porte en plus ses propres informations, sa date et sa date de retour : il devient une classe à part entière, une *classe d'association*. Décrivez, sur la `Base` fournie :

- `Livre` (table `livres`) : `id` ; `titre`, unique ; `auteur` ; `emprunts`, la liste de ses emprunts ;
- `Membre` (table `membres`) : `id` ; `nom`, unique ; `emprunts` ;
- `Emprunt` (table `emprunts`) : `id` ; `id_livre` et `id_membre`, clefs étrangères ; `date_emprunt`, une date ; `date_retour`, une date facultative (un emprunt en cours n'en a pas) ; `livre` et `membre`, l'objet de chaque côté.

Supprimez les tables existantes, puis créez celles-ci. Pourquoi un emprunt a-t-il son propre `id`, plutôt qu'une clef primaire formée de `id_livre` et `id_membre` ?

**Documentation utile**

- [Classe d'association](https://docs.sqlalchemy.org/en/21/orm/basic_relationships.html#association-object)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Une date s'annote `Mapped[date]` (`from datetime import date`), une colonne facultative `Mapped[date | None]`, et `mapped_column(unique=True)` interdit les doublons. Il y a deux relations, chacune avec ses deux côtés, qui se nomment l'un l'autre par `back_populates` : `Livre.emprunts` et `Emprunt.livre`, `Membre.emprunts` et `Emprunt.membre`.

```python
from datetime import date

from sqlalchemy import ForeignKey
from sqlalchemy.orm import Mapped, mapped_column, relationship


class Livre(Base):
  __tablename__ = "livres"
  id: Mapped[int] = mapped_column(primary_key=True)
  titre: Mapped[str] = mapped_column(unique=True)
  auteur: Mapped[str]
  emprunts: Mapped[list["Emprunt"]] = relationship(back_populates="livre")


class Membre(Base):
  ...


class Emprunt(Base):
  __tablename__ = "emprunts"
  id: Mapped[int] = mapped_column(primary_key=True)
  id_livre: Mapped[int] = mapped_column(ForeignKey(...))
  ...
  livre: Mapped[Livre] = relationship(back_populates=...)
  ...


Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)
```

### Solution

In [ ]:
from datetime import date

from sqlalchemy import ForeignKey
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship


class Base(DeclarativeBase):
  pass


class Livre(Base):
  __tablename__ = "livres"
  id: Mapped[int] = mapped_column(primary_key=True)
  titre: Mapped[str] = mapped_column(unique=True)
  auteur: Mapped[str]
  emprunts: Mapped[list["Emprunt"]] = relationship(back_populates="livre")


class Membre(Base):
  __tablename__ = "membres"
  id: Mapped[int] = mapped_column(primary_key=True)
  nom: Mapped[str] = mapped_column(unique=True)
  emprunts: Mapped[list["Emprunt"]] = relationship(back_populates="membre")


class Emprunt(Base):
  __tablename__ = "emprunts"
  id: Mapped[int] = mapped_column(primary_key=True)
  id_livre: Mapped[int] = mapped_column(ForeignKey("livres.id"))
  id_membre: Mapped[int] = mapped_column(ForeignKey("membres.id"))
  date_emprunt: Mapped[date]
  date_retour: Mapped[date | None]
  livre: Mapped[Livre] = relationship(back_populates="emprunts")
  membre: Mapped[Membre] = relationship(back_populates="emprunts")


Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# La solution part d'une Base neuve : déclarer une seconde fois une classe Livre
# sur la même Base (si vous l'avez déjà fait plus haut) lèverait une erreur.
#
# Un même membre peut emprunter plusieurs fois le même livre : le couple
# (id_livre, id_membre) n'identifie pas un emprunt, et ne peut donc pas servir de
# clef primaire. D'où un id propre à chaque emprunt.

## 2. Importer le fichier YAML

Complétez la fonction `importer_yaml(chemin, session)`, qui lit le fichier YAML `chemin` et ajoute à la session un `Livre` par livre, un `Membre` par membre et un `Emprunt` par emprunt. Les emprunts désignent livres et membres par leur titre et leur nom : gardez les objets créés dans des dictionnaires pour les retrouver. Appelez-la ensuite sur `bibliotheque.yaml` dans une session, validez, puis affichez le nombre de livres, de membres et d'emprunts de la base.

**Documentation utile**

- [`yaml.safe_load`](https://pyyaml.org/wiki/PyYAMLDocumentation)
- [`date.fromisoformat`](https://docs.python.org/fr/3/library/datetime.html#datetime.date.fromisoformat)

In [ ]:
def importer_yaml(chemin: str, session: Session) -> None:
  """Ajoute à la session les livres, membres et emprunts du fichier chemin."""
  ...

In [ ]:
# Votre code ici

### Conseils supplémentaires

`yaml.safe_load(f)` renvoie le contenu du fichier : un dictionnaire de listes. Les dates sont entre guillemets dans le fichier : ce sont des chaînes, que `date.fromisoformat` convertit, et `emprunt.get("retour")` vaut `None` pour un emprunt en cours. Il suffit d'ajouter à la session les livres, les membres et les emprunts : les relations font le lien. Vous devez obtenir 7 livres, 4 membres et 6 emprunts.

```python
import yaml


def importer_yaml(chemin: str, session: Session) -> None:
  with open(chemin, encoding="utf-8") as f:
    donnees = yaml.safe_load(f)
  livres = {
    livre["titre"]: Livre(titre=livre["titre"], auteur=livre["auteur"])
    for livre in donnees["livres"]
  }
  membres = {nom: Membre(nom=nom) for nom in donnees["membres"]}
  session.add_all(livres.values())
  session.add_all(membres.values())
  for emprunt in donnees["emprunts"]:
    retour = emprunt.get("retour")
    session.add(
      Emprunt(
        livre=...,
        membre=...,
        date_emprunt=...,
        date_retour=...,
      )
    )


with Session(engine) as session:
  importer_yaml("bibliotheque.yaml", session)
  session.commit()
  for modele in (Livre, Membre, Emprunt):
    print(modele.__name__, len(session.scalars(select(modele)).all()))
```

### Solution

In [ ]:
import yaml


def importer_yaml(chemin: str, session: Session) -> None:
  with open(chemin, encoding="utf-8") as f:
    donnees = yaml.safe_load(f)
  livres = {
    livre["titre"]: Livre(titre=livre["titre"], auteur=livre["auteur"])
    for livre in donnees["livres"]
  }
  membres = {nom: Membre(nom=nom) for nom in donnees["membres"]}
  session.add_all(livres.values())
  session.add_all(membres.values())
  for emprunt in donnees["emprunts"]:
    retour = emprunt.get("retour")
    session.add(
      Emprunt(
        livre=livres[emprunt["livre"]],
        membre=membres[emprunt["membre"]],
        date_emprunt=date.fromisoformat(emprunt["date"]),
        date_retour=date.fromisoformat(retour) if retour else None,
      )
    )


with Session(engine) as session:
  importer_yaml("bibliotheque.yaml", session)
  session.commit()
  for modele in (Livre, Membre, Emprunt):
    print(modele.__name__, len(session.scalars(select(modele)).all()))

## 3. Interroger la base

Chaque requête de cette partie s'écrit avec `select`, et s'exécute dans une session avec `session.scalars(...)`.

### Les livres empruntés par Sherlock Holmes

Affichez le titre des livres que Sherlock Holmes a empruntés, avec la date de chaque emprunt, triés par date.

**Documentation utile**

- [`Select.order_by`](https://docs.sqlalchemy.org/en/21/tutorial/data_select.html#order-by)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Partez de `select(Emprunt)`, joignez `Emprunt.membre`, filtrez sur `Membre.nom` et triez par `Emprunt.date_emprunt` avec `order_by`. Chaque emprunt donne son livre par `emprunt.livre`. Vous devez obtenir trois livres, empruntés du 5 janvier au 2 février 2026.

```python
with Session(engine) as session:
  requete = select(Emprunt).join(...).where(...).order_by(...)
  for emprunt in session.scalars(requete):
    print(emprunt.date_emprunt, emprunt.livre.titre)
```

#### Solution

In [ ]:
with Session(engine) as session:
  requete = (
    select(Emprunt)
    .join(Emprunt.membre)
    .where(Membre.nom == "Sherlock Holmes")
    .order_by(Emprunt.date_emprunt)
  )
  for emprunt in session.scalars(requete):
    print(emprunt.date_emprunt, emprunt.livre.titre)

### Les livres jamais empruntés

Affichez, par ordre alphabétique, le titre des livres qui n'ont jamais été empruntés.

**Documentation utile**

- [`any`](https://docs.sqlalchemy.org/en/21/orm/internals.html#sqlalchemy.orm.RelationshipProperty.Comparator.any)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`Livre.emprunts.any()` est vrai pour un livre qui a au moins un emprunt (en SQL, un `EXISTS`), et `~` en prend la négation. Vous devez obtenir 2 livres.

```python
with Session(engine) as session:
  requete = select(Livre).where(...).order_by(Livre.titre)
  for livre in session.scalars(requete):
    print(livre.titre)
```

#### Solution

In [ ]:
with Session(engine) as session:
  requete = select(Livre).where(~Livre.emprunts.any()).order_by(Livre.titre)
  for livre in session.scalars(requete):
    print(livre.titre)

### Les emprunts en cours

Affichez les emprunts en cours, ceux qui n'ont pas de date de retour, par date d'emprunt : la date, le titre du livre et le nom du membre.

**Documentation utile**

- [`is_`](https://docs.sqlalchemy.org/en/21/core/sqlelement.html#sqlalchemy.sql.expression.ColumnOperators.is_)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

En SQL, l'absence de valeur se teste avec `IS NULL`, que SQLAlchemy écrit `Emprunt.date_retour.is_(None)`. Vous devez obtenir 3 emprunts.

```python
with Session(engine) as session:
  requete = select(Emprunt).where(...).order_by(Emprunt.date_emprunt)
  for emprunt in session.scalars(requete):
    print(...)
```

#### Solution

In [ ]:
with Session(engine) as session:
  en_cours = Emprunt.date_retour.is_(None)
  requete = select(Emprunt).where(en_cours).order_by(Emprunt.date_emprunt)
  for emprunt in session.scalars(requete):
    print(emprunt.date_emprunt, emprunt.livre.titre, emprunt.membre.nom)

## 4. Exporter les emprunts en CSV

Complétez la fonction `exporter_csv(chemin, session)`, qui écrit tous les emprunts, par date d'emprunt, dans le fichier CSV `chemin` : une ligne par emprunt, avec les colonnes `livre`, `membre`, `date_emprunt` et `date_retour` (vide pour un emprunt en cours). Appelez-la pour écrire `emprunts.csv`, puis affichez le contenu du fichier.

**Documentation utile**

- [`csv.DictWriter`](https://docs.python.org/fr/3/library/csv.html#csv.DictWriter)
- [`date.isoformat`](https://docs.python.org/fr/3/library/datetime.html#datetime.date.isoformat)

In [ ]:
def exporter_csv(chemin: str, session: Session) -> None:
  """Écrit les emprunts de la base dans le fichier CSV chemin."""
  ...

In [ ]:
# Votre code ici

### Conseils supplémentaires

`csv.DictWriter(f, fieldnames=[...])` écrit des dictionnaires : `writeheader()` écrit l'en-tête, puis `writerow(...)` chaque ligne. Ouvrez le fichier avec `newline=""`, comme le demande le module `csv`. `isoformat()` écrit une date au format `2026-01-03`. Le fichier doit avoir 7 lignes : l'en-tête et 6 emprunts.

```python
import csv


def exporter_csv(chemin: str, session: Session) -> None:
  with open(chemin, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
      f, fieldnames=["livre", "membre", "date_emprunt", "date_retour"]
    )
    writer.writeheader()
    for emprunt in session.scalars(select(Emprunt).order_by(Emprunt.date_emprunt)):
      writer.writerow({
        "livre": emprunt.livre.titre,
        "membre": ...,
        "date_emprunt": ...,
        "date_retour": ...,
      })


with Session(engine) as session:
  exporter_csv("emprunts.csv", session)

with open("emprunts.csv", encoding="utf-8") as f:
  print(f.read())
```

### Solution

In [ ]:
import csv


def exporter_csv(chemin: str, session: Session) -> None:
  with open(chemin, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
      f, fieldnames=["livre", "membre", "date_emprunt", "date_retour"]
    )
    writer.writeheader()
    for emprunt in session.scalars(select(Emprunt).order_by(Emprunt.date_emprunt)):
      retour = emprunt.date_retour
      writer.writerow(
        {
          "livre": emprunt.livre.titre,
          "membre": emprunt.membre.nom,
          "date_emprunt": emprunt.date_emprunt.isoformat(),
          "date_retour": retour.isoformat() if retour else "",
        }
      )


with Session(engine) as session:
  exporter_csv("emprunts.csv", session)

with open("emprunts.csv", encoding="utf-8") as f:
  print(f.read())